# Truc quan hoa ket qua do an (Tieng Viet)

Notebook nay giup xem nhanh ket qua:
- Phan khuc khach hang RFM
- Luat ket hop san pham MBA

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def tim_root_du_an() -> Path:
    cwd = Path.cwd().resolve()
    ds_ung_vien = [cwd, cwd.parent]

    # Ho tro Colab: /content/<ten_repo>
    content = Path('/content')
    if content.exists():
        for p in content.iterdir():
            if p.is_dir():
                ds_ung_vien.append(p.resolve())

    for root in ds_ung_vien:
        if (root / 'src').exists() and (root / 'data').exists():
            return root

    return cwd

ROOT = tim_root_du_an()
RFM_SEGMENT_PATH = ROOT / 'data/3_curated/results/rfm/customer_segments'
RFM_SUMMARY_PATH = ROOT / 'data/3_curated/results/rfm/segment_summary'
MBA_RULES_PATH = ROOT / 'data/3_curated/results/mba/association_rules'

print('ROOT:', ROOT)
print('Co file RFM customer_segments:', RFM_SEGMENT_PATH.exists())
print('Co file RFM segment_summary:', RFM_SUMMARY_PATH.exists())
print('Co file MBA association_rules:', MBA_RULES_PATH.exists())

## 1) Doc ket qua RFM

In [ ]:
rfm_segments = None
rfm_summary = None

if RFM_SEGMENT_PATH.exists() and RFM_SUMMARY_PATH.exists():
    rfm_segments = pd.read_parquet(RFM_SEGMENT_PATH)
    rfm_summary = pd.read_parquet(RFM_SUMMARY_PATH)

    print('Da doc du lieu RFM thanh cong.')
    display(rfm_segments.head())
    display(rfm_summary.sort_values('business_score', ascending=False))
else:
    print('Chua co output RFM. Hay chay: python3 src/main_pipeline.py --project rfm --step all')

## 2) Bieu do so luong khach hang theo phan khuc

In [ ]:
if rfm_segments is not None and not rfm_segments.empty:
    bang_dem = (
        rfm_segments['segment_label']
        .value_counts()
        .rename_axis('segment_label')
        .reset_index(name='customers')
    )

    sns.barplot(
        data=bang_dem,
        x='segment_label',
        y='customers',
        hue='segment_label',
        palette='Set2',
        legend=False
    )
    plt.title('So luong khach hang theo nhan phan khuc')
    plt.xlabel('Nhan phan khuc')
    plt.ylabel('So khach hang')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print('Khong co du lieu RFM de ve bieu do.')

## 3) Bieu do chi so trung binh RFM

In [ ]:
if rfm_summary is not None and not rfm_summary.empty:
    chi_so = ['avg_recency_days', 'avg_frequency', 'avg_monetary']
    tieu_de = ['Recency trung binh (ngay)', 'Frequency trung binh', 'Monetary trung binh']

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, cot in enumerate(chi_so):
        sns.barplot(
            data=rfm_summary,
            x='segment_label',
            y=cot,
            hue='segment_label',
            palette='Set3',
            legend=False,
            ax=axes[i]
        )
        axes[i].set_title(tieu_de[i])
        axes[i].set_xlabel('Nhan phan khuc')
        axes[i].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('Khong co du lieu tong hop RFM de ve bieu do.')

## 4) Doc va xem Top luat MBA (neu co)

In [ ]:
if MBA_RULES_PATH.exists():
    mba_rules = pd.read_parquet(MBA_RULES_PATH)
    if mba_rules.empty:
        print('File association_rules ton tai nhung rong. Thu giam --min-support khi chay processing.')
    else:
        top_rules = mba_rules.sort_values('confidence', ascending=False).head(15).copy()
        top_rules['rule'] = top_rules['antecedent'].astype(str) + ' => ' + top_rules['consequent'].astype(str)
        display(top_rules[['rule', 'confidence', 'lift']])

        plt.figure(figsize=(12, 7))
        sns.barplot(
            data=top_rules,
            x='confidence',
            y='rule',
            hue='rule',
            palette='viridis',
            legend=False
        )
        plt.title('Top 15 luat ket hop theo confidence')
        plt.xlabel('Confidence')
        plt.ylabel('Luat')
        plt.tight_layout()
        plt.show()
else:
    print('Chua co output MBA. Hay chay: python3 src/main_pipeline.py --project mba --step all')